# ML-08 — Train and compare CTR models

[Open in Colab](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/work/notebooks/w05_model.ipynb)

Built on the FlyRank ML Internship dataset — https://flyrank.ai. This notebook publishes aggregate or derived diagnostics only.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').is_file()),None)
if ROOT is None:
    destination=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(destination)],check=True)
    ROOT=destination
if os.environ.get('CAPSTONE_OFFLINE') != '1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'work/requirements.txt')],check=True)
sys.path.insert(0,str(ROOT/'work/scripts'))
import json, pandas as pd, numpy as np
import ctr_capstone as study
print('Setup ready: public starter data; no token required; numerical dependencies loaded.')


Setup ready: public starter data; no token required; numerical dependencies loaded.


## Run the repeatable analysis

Execute the full model fit, client split, validation selection, final comparison, and derived queue.

In [2]:
r=study.run()

{
  "selected_method": "random_forest",
  "eligible_rows": 16726,
  "splits": {
    "train": {
      "rows": 8483,
      "clients": 17
    },
    "validation": {
      "rows": 4330,
      "clients": 5
    },
    "test": {
      "rows": 3913,
      "clients": 6
    }
  },
  "test": {
    "position_baseline": {
      "mae_pp": 0.2013234607495584,
      "weighted_mae_pp": 0.2042125527753737,
      "rmse_pp": 0.2870019252129417,
      "observed_ctr_pp": 0.2850271116236826,
      "predicted_ctr_pp": 0.33993391818137225,
      "n": 3913
    },
    "random_forest": {
      "mae_pp": 0.172949732444668,
      "weighted_mae_pp": 0.17261499349796033,
      "rmse_pp": 0.2742262173661863,
      "observed_ctr_pp": 0.2850271116236826,
      "predicted_ctr_pp": 0.26412428668911,
      "n": 3913
    },
    "hist_gradient_boosting": {
      "mae_pp": 0.17107748554758137,
      "weighted_mae_pp": 0.17499545088407836,
      "rmse_pp": 0.27630515157474955,
      "observed_ctr_pp": 0.2850271116236826,
     

## Comparison table

Random Forest is selected by validation weighted MAE. Report every method even where the secondary metric prefers another.

In [3]:
print(pd.DataFrame(r['test']).T[['mae_pp','weighted_mae_pp','rmse_pp']].to_string())
print('Selected before test:',r['selected_method'])

                          mae_pp  weighted_mae_pp   rmse_pp
position_baseline       0.201323         0.204213  0.287002
random_forest           0.172950         0.172615  0.274226
hist_gradient_boosting  0.171077         0.174995  0.276305
Selected before test: random_forest


## Errors and interpretation

Three rounded diagnostics show individual-page misses without raw identifying records.

In [4]:
print(pd.DataFrame(r['wrong_cases']).to_string(index=False))

diagnostic_case position_band  observed_ctr_pp  predicted_ctr_pp                                                                                      explanation
              A         11–20            5.426             0.275 Large residual on an individual page; query mix, brand demand and snippet intent are unmeasured.
              B           1–3            5.194             0.492 Large residual on an individual page; query mix, brand demand and snippet intent are unmeasured.
              C          4–10            2.541             0.274 Large residual on an individual page; query mix, brand demand and snippet intent are unmeasured.
